# 05.3 Knowledge Distillation for LLM InferenceComparing distillation approaches: SwiftKV (prefill skip), Caprese (quality recovery),Llamba (cross-architecture transfer), and head-to-head with quantization & speculative decoding.

In [ ]:
import syssys.path.insert(0, '../../..')import numpy as npimport matplotlib.pyplot as pltfrom content.utils.benchmark import BenchmarkTimerfrom content.utils.latency import LatencyTracker

## SwiftKV: Prefill Layer Skip SimulationSwiftKV distills a full model into one that skips KV computation in later layers during prefill,reusing earlier layers' KV projections. We simulate the latency savings.

In [ ]:
# SwiftKV prefill skip simulationnum_layers = 32hidden_dim = 4096seq_len = 2048batch_size = 4# Full prefill: all layers compute KVfull_flops_per_layer = 2 * seq_len * hidden_dim * hidden_dim * 2  # Q,K,V projectionsfull_prefill_flops = full_flops_per_layer * num_layers# SwiftKV: skip KV in last N layers (reuse from earlier layers)skip_fractions = [0.25, 0.5, 0.75]results_swiftkv = {}for skip_frac in skip_fractions:    skipped = int(num_layers * skip_frac)    # Skipped layers only compute Q (save 2/3 of attention proj flops)    active_flops = full_flops_per_layer * (num_layers - skipped)    skip_flops = (full_flops_per_layer / 3) * skipped  # Q only    total = active_flops + skip_flops    results_swiftkv[f"{int(skip_frac*100)}%"] = {        "speedup": full_prefill_flops / total,        "flops_saved_pct": (1 - total / full_prefill_flops) * 100    }    print(f"Skip {int(skip_frac*100)}% layers: {results_swiftkv[f'{int(skip_frac*100)}%']['speedup']:.2f}x speedup, "          f"{results_swiftkv[f'{int(skip_frac*100)}%']['flops_saved_pct']:.1f}% FLOPs saved")

## Caprese: Quality Recovery via Continued DistillationAfter aggressive compression, Caprese applies layer-wise distillation with a quality recoveryphase. We model the quality-compute tradeoff.

In [ ]:
# Caprese quality recovery simulation# Model: distilled student starts at lower quality, recovers with more training tokenstraining_tokens_B = np.array([0, 10, 50, 100, 200, 500])teacher_quality = 82.0  # benchmark score# Different compression levelscompressions = {    "2x (16 layers)": {"initial_quality": 72.0, "recovery_rate": 0.025},    "3x (11 layers)": {"initial_quality": 65.0, "recovery_rate": 0.018},    "4x (8 layers)":  {"initial_quality": 58.0, "recovery_rate": 0.012},}plt.figure(figsize=(10, 6))for label, params in compressions.items():    quality = params["initial_quality"] + (teacher_quality - params["initial_quality"]) * (        1 - np.exp(-params["recovery_rate"] * training_tokens_B))    plt.plot(training_tokens_B, quality, 'o-', label=label, linewidth=2)plt.axhline(y=teacher_quality, color='k', linestyle='--', alpha=0.5, label='Teacher')plt.xlabel("Distillation Tokens (B)")plt.ylabel("Benchmark Score")plt.title("Caprese: Quality Recovery vs Training Compute")plt.legend()plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## Llamba: Cross-Architecture DistillationLlamba distills a Transformer teacher into a recurrent (Mamba-based) student,trading attention's O(n²) for O(n) inference with minimal quality loss.

In [ ]:
# Llamba: Transformer -> Mamba latency modelseq_lengths = np.array([512, 1024, 2048, 4096, 8192, 16384])# Transformer: O(n^2) attention dominates at long sequencestransformer_latency_ms = 5 + 0.0001 * seq_lengths**2 / 1000  # quadratic component# Mamba student: O(n) linear scanmamba_latency_ms = 3 + 0.008 * seq_lengths  # linear component# Quality retention (Llamba reports ~95-97% of teacher on most benchmarks)quality_retention = 0.96plt.figure(figsize=(10, 5))plt.subplot(1, 2, 1)plt.plot(seq_lengths, transformer_latency_ms, 's-', label='Transformer Teacher', linewidth=2)plt.plot(seq_lengths, mamba_latency_ms, 'o-', label='Mamba Student (Llamba)', linewidth=2)plt.xlabel("Sequence Length")plt.ylabel("Decode Latency (ms/token)")plt.title("Latency Scaling: Transformer vs Llamba")plt.legend()plt.grid(True, alpha=0.3)plt.subplot(1, 2, 2)speedup = transformer_latency_ms / mamba_latency_msplt.bar(range(len(seq_lengths)), speedup, color='#2563eb', alpha=0.7)plt.xticks(range(len(seq_lengths)), [str(s) for s in seq_lengths], rotation=45)plt.xlabel("Sequence Length")plt.ylabel("Speedup (x)")plt.title("Llamba Speedup vs Sequence Length")plt.grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.show()print(f"Quality retention: {quality_retention*100:.0f}% of teacher benchmarks")print(f"Max speedup at seq_len={seq_lengths[-1]}: {speedup[-1]:.1f}x")

## Comparison: Distillation vs Quantization vs Speculative DecodingEach technique trades off differently across latency, throughput, quality, and memory.

In [ ]:
# Head-to-head comparison on a 7B model baselinetechniques = {    "Baseline (FP16)":       {"latency": 1.0, "throughput": 1.0, "quality": 1.0, "memory": 1.0},    "INT4 Quantization":     {"latency": 0.55, "throughput": 1.8, "quality": 0.97, "memory": 0.30},    "Spec Decoding (draft)": {"latency": 0.40, "throughput": 1.0, "quality": 1.0, "memory": 1.35},    "SwiftKV Distill":       {"latency": 0.60, "throughput": 1.5, "quality": 0.95, "memory": 0.85},    "Llamba (Mamba)":        {"latency": 0.35, "throughput": 2.2, "quality": 0.96, "memory": 0.70},    "Quant + Distill":       {"latency": 0.35, "throughput": 2.5, "quality": 0.93, "memory": 0.25},}# Radar-style comparison tableprint(f"{'Technique':<25} {'Latency':>8} {'Throughput':>11} {'Quality':>8} {'Memory':>8}")print("-" * 65)for name, metrics in techniques.items():    print(f"{name:<25} {metrics['latency']:>7.2f}x {metrics['throughput']:>10.1f}x "          f"{metrics['quality']:>7.1%} {metrics['memory']:>7.2f}x")

In [ ]:
# Visualization: normalized comparisonfig, ax = plt.subplots(figsize=(12, 6))names = list(techniques.keys())[1:]  # skip baselinemetrics_names = ["latency", "throughput", "quality", "memory"]x = np.arange(len(names))width = 0.2for i, metric in enumerate(metrics_names):    values = [techniques[n][metric] for n in names]    bars = ax.bar(x + i * width, values, width, label=metric.capitalize(), alpha=0.8)ax.set_xticks(x + width * 1.5)ax.set_xticklabels(names, rotation=20, ha='right')ax.set_ylabel("Relative to Baseline (1.0)")ax.set_title("Inference Optimization Techniques: Multi-Dimensional Comparison")ax.legend()ax.axhline(y=1.0, color='k', linestyle='--', alpha=0.3)ax.grid(True, alpha=0.2, axis='y')plt.tight_layout()plt.show()

## Key Takeaways| Technique | Best For | Limitation ||-----------|----------|------------|| **SwiftKV** | Prefill-bound workloads (long prompts) | Requires distillation training budget || **Caprese** | Recovering quality after aggressive pruning | High token cost for full recovery || **Llamba** | Long-sequence inference (>4K tokens) | Architecture change, retraining needed || **Quantization** | Memory-constrained deployment | Quality degrades below 4-bit || **Spec Decoding** | Latency-sensitive, quality-critical | Extra memory for draft model || **Combined** | Maximum efficiency | Compound quality loss risk |**Recommendation**: Start with quantization (cheapest), add distillation for further gains,use speculative decoding only when quality cannot be compromised.